In [1]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append('../scripts/')
import monday_board_update
import requests

In [2]:
parent_dir = Path("/Users/hinashah/Documents/HEAL/DataPackageMondayBoard/")
## Get either progress tracker or index values - Could use both. 

In [3]:
# Read the Monday Board
monday_bd_file = "Data_package_tracking_1761766145.xlsx"
monday_bd = parent_dir / monday_bd_file
monday_df = monday_board_update.import_monday_board(parent_dir, file_name=monday_bd_file, rows_to_skip=2)
print(len(monday_df))

[PosixPath('/Users/hinashah/Documents/HEAL/DataPackageMondayBoard/Data_package_tracking_1761766145.xlsx')]
Index(['Name', 'Repository Link', 'Data package assessment result',
       'Date Reviewed', 'Repository', 'Reviewed By', 'Access Type',
       'Data Type', 'Platform's assessment', 'Data Quality Comments',
       'SLMD in Data Deposit', 'VLMD in Data Deposit',
       'Actual Data in Data Deposit', 'Code/scripts in Data Deposit',
       'Publications/DOI in Data Deposit', 'Protocol document in Data Deposit',
       'Data deposit usability', 'Permission to share as an example',
       'Permission to use as an example in presentations',
       'Present in Webinar', 'Response Date', 'Contacted PI Name',
       'Contacted PI Email', 'HEAL Compliant Repository?', 'Data Embargoed?',
       'Intake per DD tracker (MIRROR)',
       'SME Verification per DD tracker (MIRROR)',
       'DD Transformation status per DD Tracker (MIRROR)',
       'Data Dictionary Tracker entry', 'HEAL Studies',
 

In [4]:
## Redo the code to start with either progress tracker table or MDS itself. Probably MDS would be fine. DO NOT START WITH HSS. HSS updates would be slower. 
DEFAULT_MDS_ENDPOINT = 'https://healdata.org/mds/metadata'
MDS_DEFAULT_LIMIT = 10000
all_studies_dict = []
HEAL_STUDY_GUID_TYPES = [
    'discovery_metadata',                   # Fully registered studies.
    'unregistered_discovery_metadata'       # Studies added to the Platform MDS but without the investigator registering the study.
]

metadata_ids = []
for heal_study_guid_type in HEAL_STUDY_GUID_TYPES:
    result = requests.get(DEFAULT_MDS_ENDPOINT, params={
        '_guid_type': heal_study_guid_type,
        'limit': MDS_DEFAULT_LIMIT,
    })
    if not result.ok:
        print(f'Could not retrieve metadata list for guid_type {heal_study_guid_type}: {result}')
    metadata_ids.extend(result.json())
study_ids = list(metadata_ids)
print(f"Getting information for {len(study_ids)} studies from HEAL MDS")

Getting information for 1140 studies from HEAL MDS


In [5]:
## Try to use HSS API to get to HEAL Data
DEFAULT_HSS_ENDPOINT = 'https://heal-dev.apps.renci.org/search-api'
study_query = {'query' : '*', 'size' : 2000 }
variable_query = {"query":"*","filters":[{"field":"metadata.cde_mapping","operator":"size_gt","value":0}],"size":1000,"offset":0}

studies_result = requests.post(DEFAULT_HSS_ENDPOINT+'/studies', json=study_query)
if not studies_result.ok:
    print(f"Error retrieving study data from HSS")
    study_data = None
else:
    study_data = {k['id'] : k for k in studies_result.json()['results']}
    print(f"Studies endpoing returned {len(study_data)} elements")

variables_result = requests.post(DEFAULT_HSS_ENDPOINT+'/variables', json=variable_query)
if not variables_result.ok:
    print(f"Error retrieving variables data")
    variable_data = None
else:
    variable_data = {k['id'] : k for k in variables_result.json()['results']}
    print(f"Variables endpoing returned {len(variable_data)} elements")

Studies endpoing returned 1144 elements
Variables endpoing returned 424 elements


In [6]:
## Organize the variable data by studies
variable_to_cde = dict()
for variable in variable_data:
    parents = variable_data[variable]['parents']
    for p in parents:
        if p not in variable_to_cde:
            variable_to_cde[p] = []
        variable_to_cde[p].append((variable, variable_data[variable]['metadata']['cde_mapping']['measure']))
print(len([k for k in variable_to_cde.keys() if k.startswith('HDP')]))

26


In [7]:
# ## Get HEAL Indices data.
# ## Read the index download, and create a dataset with package links using information from Platform MDS
# import json
# index_v2 = Path(parent_dir / "DataReuseSummaries/heal-dev-studies-index.json")
# with open(index_v2, "r") as f:
#     all_lines = f.readlines()
#     all_elements_1 = [json.loads(s) for s in all_lines]
# index_data = {s["_id"]:s for s in all_elements_1}
# print(f"HSS studies index data has: {len(index_data)} elements")

In [8]:
cde_col_names = set()

for study_id in study_ids:
    # start creating the data structure
    new_s = study_data[study_id] if study_id in study_data else None
    repos =new_s['metadata']['Data Package Links'] if new_s and 'Data Package Links' in new_s['metadata'] else []
    
    num_mappings = 0
    list_cdes = set()
    if study_id in variable_to_cde:
        num_mappings = len(variable_to_cde[study_id])
        list_cdes = set(k[1] for k in variable_to_cde[study_id])

    d = {
         'id':study_id, 
         'num_variables':len(new_s['variable_list']) if new_s else 0, # index
         'num_cdes':len(new_s['section_list']) if new_s else 0, # index
         'data_available': new_s['metadata']['Data Availability'] if new_s and 'Data Availability' in new_s['metadata'] else '', #index, and platform data assessment
         'cdes': new_s['section_list'] if new_s else [], #index
         'num_variable_cde_mappings': num_mappings,
         'variable_cde_mappings': list_cdes
        }


    # Create a cde usage indicator data structure in the form of python dict
    cde_dict = {k:1 for k in d['cdes']}
    cde_col_names = cde_col_names.union(set(cde_dict.keys()))

    # Get links from platform that were assessed to be data, and "other_study_websites" from MDS could be something else.
    # We will add these to the data frame as individual links to be matched with platform. 
    # We will add links from the list of other websites only if they are not duplicate from the data package assessment links
    mds_response = requests.get(DEFAULT_MDS_ENDPOINT + '/' + study_id)
    study_json = mds_response.json()
    gen3_discovery = study_json.get('gen3_discovery', None)
    study_metadata = gen3_discovery.get('study_metadata', None)
    repositories = []
    if study_metadata is not None and ('metadata_location' in study_metadata and 'data_repositories' in study_metadata['metadata_location']):
            repositories = study_metadata['metadata_location']['data_repositories']
    other_website_links = []
    if study_metadata is not None and ('metadata_location' in study_metadata and 'other_study_websites' in study_metadata['metadata_location']):
        other_website_links = study_metadata['metadata_location']['other_study_websites']
    repo_links = [r['repository_study_link'] for r in repositories if 'repository_study_link' in r and len(r['repository_study_link']) > 0]
    other_links = [r for r in other_website_links if r not in repo_links]

    num_repos = len(repo_links) + len(other_links)
    num_data_repos = len(repo_links)
    num_other_repos = len(other_links)

    if len(repo_links) > 0 or len(other_links) > 0:
        # print(repos)
        cnt = 0 
        if len(repo_links) > 0:
            for r in repositories:
                if 'repository_study_link' in r and len(r['repository_study_link']) > 0:
                    all_studies_dict.append({**d, 
                                             'repo_link': r['repository_study_link'], 
                                             'repo_name': r['repository_name'], 
                                             'num_repositories':num_repos, 
                                             'num_data_repositories': num_data_repos, 
                                             'num_other_links': num_other_repos, 
                                             'is_other_website': False, 
                                             **cde_dict})
                    cnt +=1
            if cnt == 0:
                print(f"ERROR: Check HDPID {k} for length of repositories, didnt add any to the dd {len(repositories)}")

        if len(other_links) > 0:
            for r in other_links:
                all_studies_dict.append({**d, 
                                         'repo_link': r, 
                                         'repo_name': '', 
                                         'num_repositories':num_repos, 
                                         'num_data_repositories': num_data_repos, 
                                         'num_other_links': num_other_repos, 
                                         'is_other_website': True, 
                                         **cde_dict})
    else:
            all_studies_dict.append({**d, 
                                     'repo_link': '', 
                                     'repo_name': '', 
                                     'num_repositories': 0, 
                                     'num_other_links': 0,
                                     'num_data_repositories': 0,
                                     **cde_dict })

all_studies_pd = pd.DataFrame(all_studies_dict)
all_studies_pd.to_csv(parent_dir/"DataReuseSummaries/studies_index.csv", index=False)
all_studies_pd['num_repositories'].describe()
print(len(all_studies_dict))
print(f"Number of HDPIDs {len(all_studies_pd['id'].unique())}")


1213
Number of HDPIDs 1140


In [ ]:
# cde_col_names = set()

# for study_id in study_ids:
#     # start creating the data structure
#     new_s = index_data[study_id]['_source'] if study_id in index_data else None
#     repos =new_s['metadata']['Data Package Links'] if new_s and 'Data Package Links' in new_s['metadata'] else []
    
#     d = {
#          'id':study_id, 
#          'num_variables':len(new_s['variable_list']) if new_s else 0, # index
#          'num_cdes':len(new_s['section_list']) if new_s else 0, # index
#          'data_available': new_s['metadata']['Data Availability'] if new_s and 'Data Availability' in new_s['metadata'] else '', #index, and platform data assessment
#          'cdes': new_s['section_list'] if new_s else [] #index
#         }

#     # Create a cde usage indicator data structure in the form of python dict
#     cde_dict = {k:1 for k in d['cdes']}
#     cde_col_names = cde_col_names.union(set(cde_dict.keys()))

#     # Get links from platform that were assessed to be data, and "other_study_websites" from MDS could be something else.
#     # We will add these to the data frame as individual links to be matched with platform. 
#     # We will add links from the list of other websites only if they are not duplicate from the data package assessment links
#     mds_response = requests.get(DEFAULT_MDS_ENDPOINT + '/' + study_id)
#     study_json = mds_response.json()
#     gen3_discovery = study_json.get('gen3_discovery', None)
#     study_metadata = gen3_discovery.get('study_metadata', None)
#     repositories = []
#     if study_metadata is not None and ('metadata_location' in study_metadata and 'data_repositories' in study_metadata['metadata_location']):
#             repositories = study_metadata['metadata_location']['data_repositories']
#     other_website_links = []
#     if study_metadata is not None and ('metadata_location' in study_metadata and 'other_study_websites' in study_metadata['metadata_location']):
#         other_website_links = study_metadata['metadata_location']['other_study_websites']
#     repo_links = [r['repository_study_link'] for r in repositories if 'repository_study_link' in r and len(r['repository_study_link']) > 0]
#     other_links = [r for r in other_website_links if r not in repo_links]

#     num_repos = len(repo_links) + len(other_links)
#     num_data_repos = len(repo_links)
#     num_other_repos = len(other_links)

#     if len(repo_links) > 0 or len(other_links) > 0:
#         # print(repos)
#         cnt = 0 
#         if len(repo_links) > 0:
#             for r in repositories:
#                 if 'repository_study_link' in r and len(r['repository_study_link']) > 0:
#                     all_studies_dict.append({**d, 
#                                              'repo_link': r['repository_study_link'], 
#                                              'repo_name': r['repository_name'], 
#                                              'num_repositories':num_repos, 
#                                              'num_data_repositories': num_data_repos, 
#                                              'num_other_links': num_other_repos, 
#                                              'is_other_website': False, 
#                                              **cde_dict})
#                     cnt +=1
#             if cnt == 0:
#                 print(f"ERROR: Check HDPID {k} for length of repositories, didnt add any to the dd {len(repositories)}")

#         if len(other_links) > 0:
#             for r in other_links:
#                 all_studies_dict.append({**d, 
#                                          'repo_link': r, 
#                                          'repo_name': '', 
#                                          'num_repositories':num_repos, 
#                                          'num_data_repositories': num_data_repos, 
#                                          'num_other_links': num_other_repos, 
#                                          'is_other_website': True, 
#                                          **cde_dict})
#     else:
#             all_studies_dict.append({**d, 
#                                      'repo_link': '', 
#                                      'repo_name': '', 
#                                      'num_repositories': 0, 
#                                      'num_other_links': 0,
#                                      'num_data_repositories': 0,
#                                      **cde_dict })

# all_studies_pd = pd.DataFrame(all_studies_dict)
# all_studies_pd.to_csv(parent_dir/"DataReuseSummaries/studies_index.csv", index=False)
# all_studies_pd['num_repositories'].describe()
# print(len(all_studies_dict))
# print(f"Number of HDPIDs {len(all_studies_pd['id'].unique())}")


1214
Number of HDPIDs 1140


In [10]:
## Read in platform curated digital asset tracking sheet. 
platform_assessments_df = pd.read_csv(parent_dir/"HEAL Data & Digital Assets Tracking Mar 2026.csv")
print(platform_assessments_df.columns)

platform_assessment_dicts = []
ind_list = [f"{k}" for k in range(1,6)]

# These are the types of links that we will be looking at.
# As of now, we will not be incorporating "Awaiting Data" assessments on the Package Board or package assessments.
valid_assessments = ['Data', 'Non-data asset', 'Peer-reviewed publication', 'Code']
make_repolink_dict = lambda repo_link, hdp_id, assessment : {'repository_study_link':'' if pd.isna(repo_link) else repo_link.strip(), 
                                                             'hdp_id':hdp_id.strip(), 
                                                             'platform_assessment': assessment.strip() if (not pd.isna(assessment) and assessment.strip() in valid_assessments) else ''
                                                            }

for row in platform_assessments_df.iterrows():
    for ind in ind_list:
        if not pd.isna(row[1][f"URL / DOI {ind}"]):
            platform_assessment_dicts.append(make_repolink_dict(row[1][f"URL / DOI {ind}"], row[1][f"HDP_ID"], row[1][f"Determination {ind}"]))

print(len(platform_assessment_dicts))
platform_assessments_subdf = pd.DataFrame(platform_assessment_dicts)
platform_assessments_subdf = platform_assessments_subdf[platform_assessments_subdf.platform_assessment.isin(valid_assessments)]
num_repos = platform_assessments_subdf['hdp_id'].value_counts()
platform_assessments_subdf = pd.merge(platform_assessments_subdf, num_repos, how='left', left_on='hdp_id', right_on='hdp_id')
platform_assessments_subdf.rename(columns={'count':'platform_num_repositories'}, inplace=True)
platform_assessments_subdf['platform_assessment'].value_counts()

Index(['HDP_ID', 'Study Title', 'Repository', 'URL / DOI 1', 'Determination 1',
       'Action 1', 'URL / DOI 2', 'Determination 2', 'Action 2', 'URL / DOI 3',
       'Determination 3', 'Action 3', 'URL / DOI 4', 'Determination 4',
       'Action 4', 'URL / DOI 5', 'Determination 5', 'Action 5', 'Added by:',
       'Date Added:', 'Date Last Updated:', 'Notes', 'Automatic Datestamps'],
      dtype='object')
216


platform_assessment
Data                         141
Non-data asset                 8
Peer-reviewed publication      3
Code                           2
Name: count, dtype: int64

In [11]:
## Start matching links between index/MDS to the platform assessment sheet.
# Same targets can have different looking links on Platform assessment sheet and MDS, hence we need this piece of code to match them

all_studies_pd_links = all_studies_pd[all_studies_pd.num_repositories>0] 
print(len(all_studies_pd_links))

# Do an outer join on all MDS links and platform links
w_platform_links = pd.merge(all_studies_pd_links, platform_assessments_subdf, how = 'outer', left_on = 'id', right_on='hdp_id')
w_platform_links.rename(columns={'id':'index_hdp_id', 'hdp_id':'platform_hdp_id'}, inplace=True)
print(len(w_platform_links))

## Iterate through the rows and find the matches.

# Mark exact matches tot he URL or links that do not exist in NDS. This may happen if the link has not yet been published to MDS.
w_platform_links['matches'] = ['True New Data' if (pd.isna(prl) and pa in valid_assessments) else ('True Exact Match' if prl==irl else 'False') for prl, irl, pa in w_platform_links[['repo_link', 'repository_study_link', 'platform_assessment']].values]
num_matches = w_platform_links[['platform_hdp_id','matches']][w_platform_links['matches'] == 'True Exact Match'].value_counts()
w_platform_links = pd.merge(w_platform_links, num_matches, how='left', left_on='index_hdp_id', right_on='platform_hdp_id')
w_platform_links['count'].fillna(0, inplace=True)
w_platform_links.rename(columns={'count':'match_count'}, inplace=True)
w_platform_links.to_csv(parent_dir/"DataReuseSummaries/platform_link_match_v0.csv", index=False)

corrected_matches = []
for index, row in  w_platform_links.iterrows():
    if pd.isna(row.repo_link): ## Chances are this is a new data link (MDS Repo link does not exist)
        if row.matches != 'True New Data':
            print(f"Something wrong with {row.index_hdp_id}")
        corrected_matches.append(row.matches)
        continue
    if pd.isna(row.repository_study_link):
        if row.match_count != 0:
            print(f"Something wrong with {row.index_hdp_id}")
        corrected_matches.append(row.matches)
        continue
    if row.match_count == row.platform_num_repositories: # Each platform repository has a corresponding match in the list of MDS links
        corrected_matches.append(row.matches)
        continue
    ## Looking at inconsistencies:
    # If there's only one repo both on MDS and asset tracking - it's very likely that it's the same 
    if row.num_repositories == 1 and row.platform_num_repositories == 1:
        corrected_matches.append("Potential Match")
        continue
    
    index_url = row.repo_link.replace("https://","").replace("doi.org","").replace("www.", "")
    platform_url = row.repository_study_link.replace("https://","").replace("doi.org","").replace("www.", "")
    if index_url in platform_url or platform_url in index_url:
        corrected_matches.append("Potential Match")
    else:
        corrected_matches.append(row.matches)

w_platform_links['corrected_matches'] = corrected_matches
# w_platform_links['matches'] = matches
w_platform_links.to_csv(parent_dir/"DataReuseSummaries/platform_link_match.csv", index=False)
platform_matches = w_platform_links[w_platform_links.corrected_matches.isin(['Potential Match', 'True Exact Match'])].copy(deep=True)
print(len(platform_matches))

# Combine platofrm and index studies with matched links.
index_platform_df = pd.merge(all_studies_pd, platform_matches[['index_hdp_id', 'repo_link', 'platform_assessment']], how='left', left_on=['id', 'repo_link'], right_on  = ['index_hdp_id', 'repo_link'])
print("Before dropping")
print(len(index_platform_df))
print(f"Number of HDPIDs {len(index_platform_df['id'].unique())}")
print(f"Number of Studies with a repository link: {len(index_platform_df[index_platform_df.num_repositories > 0]['id'].unique())}")

## Remove repository links that are marked as other websites, and do not have a corresponding platform assessment
to_drop = [ True if ( pd.isna(pa) and (iso is True) and ( nr > 1 ) and (nd >= 1) ) else False for (pa, iso, nr, nd) in index_platform_df[['platform_assessment', 'is_other_website', 'num_repositories', 'num_data_repositories']].values]
index_platform_df['to_drop'] = to_drop
to_drop_rows = index_platform_df[pd.isna(index_platform_df['platform_assessment']) & (index_platform_df['is_other_website'] == True)]
index_platform_df.loc[pd.isna(index_platform_df['platform_assessment']) & 
                      (index_platform_df['is_other_website'] == True) & 
                      (index_platform_df['num_repositories'] == 1), 
                      'repo_link'] = ''
import numpy as np
index_platform_df.loc[pd.isna(index_platform_df['platform_assessment']) & 
                      (index_platform_df['is_other_website'] == True) &
                      index_platform_df['num_repositories'] == 1, 'is_other_website'] = np.nan

index_platform_df = index_platform_df[index_platform_df['to_drop'] == False]

repo_links = index_platform_df[ index_platform_df['repo_link']!=''][['id', 'repo_link']].drop_duplicates()
link_counts = repo_links['id'].value_counts()
index_platform_df.num_repositories = [link_counts[k] if k in link_counts else 0 for k in index_platform_df['id']]

# to_drop_rows =  index_platform_df[pd.isna(index_platform_df['platform_assessment']) & index_platform_df['is_other_website'] == True]
# print(len(to_drop_rows))
# index_platform_df.drop(labels=to_drop_rows.index, inplace=True)

print("**** After dropping")

print(f"Number of HDPIDs {len(index_platform_df['id'].unique())}")
print(f"Number of Studies with a repository link: {len(index_platform_df[index_platform_df['repo_link']!='']['id'].unique())}")
index_platform_df.to_csv(parent_dir/"DataReuseSummaries/index_platform_matches.csv", index=False)

## Add Urja's assessments
manual_assessments = pd.read_csv(parent_dir/"DataReuseSummaries/Urja_data_package_assessments.csv")
index_platform_manual = pd.merge(index_platform_df, manual_assessments[['Name', 'Repository Link', "Urja's comments", "Urja Usability Assessment", "CDE DD mappings"]], how='left', left_on=['id', 'repo_link'], right_on=['Name', 'Repository Link'])
index_platform_manual.drop(columns =['Name', 'Repository Link','index_hdp_id'], inplace=True)
index_platform_manual.rename(columns={"Urja's comments":"Manual assessment notes", "Urja Usability Assessment":"Manual assessment"}, inplace=True)
## What else for data package assessments?
col_order = [k for k in index_platform_manual.columns if k not in cde_col_names]
print(cde_col_names)
col_order.extend(cde_col_names)
print(col_order)

index_platform_manual = index_platform_manual[col_order]
index_platform_manual.to_csv(parent_dir/"DataReuseSummaries/index_platform_matches_manual.csv", index=False)
## Need CDE mappings from somewhere - probably index itself.

232
424
155
Before dropping
1213
Number of HDPIDs 1140
Number of Studies with a repository link: 159
**** After dropping
Number of HDPIDs 1140
Number of Studies with a repository link: 126
{'patient-health-questionnaire-8', 'fabq-physical-activity', 'pgic', 'keele-start-back', 'physical-function-6b', 'dc-tmd-exam-form', 'promis-self-efficacy-managing-emotions-4a', 'pcs-parent', 'domain-specific-life-satisfaction-18', 'promis-pediatric-depressive-symptoms-8a', 'mypas', 'cgic', 'msas-sf', 'opioid-risk-tool', 'gad2', 'pediatric-demographic', 'whoqol-bref', 'promis-pediatric-mobility-8a', 'promis-satisfaction-with-social-roles-8a', 'promis-anxiety-sf-4a', 'sleep-disturbance-6a', 'promis-pain-interference-8a', 'promis-pain-interference-cat', '30-second-chair-stand-assessment', 'promis-fatigue-7b', 'promis-severity-of-substance-use-7a', 'scq', 'feges', 'maia-2', 'psqi', 'fabq', 'promis-physical-functional-cat', 'pbhq-as', 'promis-emotional-support-4a', 'audit-self-report', 'pips', 'osu-tbi',

/var/folders/tb/c_qhpk1j2jj3knzfnw72yj080000gq/T/ipykernel_19300/59770035.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  w_platform_links['count'].fillna(0, inplace=True)


In [12]:
## Combine data and show stats
print(f"Number of HDPIDs {len(index_platform_manual['id'].unique())}")
print(f"Number of Studies with a repository link: {len(index_platform_manual[index_platform_manual.num_repositories > 0]['id'].unique())}")
print(f"Number of Repository links in these studies: {len(index_platform_manual[index_platform_manual.num_repositories > 0])}")
studies_with_data_by_platform = index_platform_manual[index_platform_manual.platform_assessment == 'Data']['id'].unique()
print(f"Number of studies with data according to platform's assessment: {len(studies_with_data_by_platform)}")
vlmd_available = index_platform_manual[index_platform_manual.id.isin(studies_with_data_by_platform) & (index_platform_manual.num_variables > 0)]['id'].unique()
cde_available = index_platform_manual[index_platform_manual.id.isin(studies_with_data_by_platform) & (index_platform_manual.num_cdes > 0)]['id'].unique()
print(f"Out of these, we have VLMDs for {len(vlmd_available)} and we have CDEs for {len(cde_available)} ")
vlmd_cde_available = index_platform_manual[index_platform_manual.id.isin(studies_with_data_by_platform) & 
                                           (index_platform_manual.num_cdes > 0) & 
                                           (index_platform_manual.num_variables > 0)]['id'].unique()
print(f"We have CDEs and VLMDs for {len(vlmd_cde_available)}")

usability_notes = index_platform_manual[
    index_platform_manual.id.isin(studies_with_data_by_platform) & 
    (index_platform_manual.num_cdes > 0) & 
    (index_platform_manual.num_variables > 0) & 
    (~pd.isna(index_platform_manual["Manual assessment"]))
    ]
print(f"From the above, number of packages for which usability Notes are available for: {len(usability_notes)}")
freq = usability_notes["Manual assessment"].value_counts()
print("-------------------------------")
print(f"Usability frequencies: {freq}")
print("-------------------------------")
freq_pd = index_platform_manual[~pd.isna(index_platform_manual["Manual assessment"])]
print(f"Usability notes available for in general: {len(freq_pd)}")
freq = index_platform_manual[~pd.isna(index_platform_manual["Manual assessment"])]["Manual assessment"].value_counts()
print("-------------------------------")
print(f"Usability frequencies: {freq}")
print("-------------------------------")


Number of HDPIDs 1140
Number of Studies with a repository link: 126
Number of Repository links in these studies: 179
Number of studies with data according to platform's assessment: 114
Out of these, we have VLMDs for 36 and we have CDEs for 34 
We have CDEs and VLMDs for 25
From the above, number of packages for which usability Notes are available for: 5
-------------------------------
Usability frequencies: Manual assessment
Usable for Secondary Analysis    3
Need to contact PI               2
Name: count, dtype: int64
-------------------------------
Usability notes available for in general: 35
-------------------------------
Usability frequencies: Manual assessment
Usable for Secondary Analysis    14
Need to contact PI               13
No usable Data                    8
Name: count, dtype: int64
-------------------------------


In [13]:
index_platform_manual.repo_link

0                                                   
1                 https://doi.org/10.26275/aexe-anbk
2       https://doi.org/10.6084/m9.figshare.25036364
3       https://doi.org/10.6084/m9.figshare.25036925
4       https://doi.org/10.6084/m9.figshare.24867198
                            ...                     
1188                                                
1189                                                
1190                                                
1191                                                
1192                                                
Name: repo_link, Length: 1193, dtype: object

In [ ]:
## Create a dataset to be imported into Monday board.
monday_doc = index_platform_manual[~pd.isna(index_platform_manual.repo_link) & (index_platform_manual.repo_link != "")][['id', 'num_variables', 'num_cdes', 'repo_link', 'repo_name', 'platform_assessment']].copy(deep=True)
print(len(monday_doc))
monday_doc['VLMD on Platform'] = ['Yes' if n > 0 else 'No' for n in monday_doc['num_variables'].values]
monday_doc['CDE mapping on HSS'] = ['Yes' if n > 0 else 'No' for n in monday_doc['num_cdes'].values]
print(monday_doc.columns)
monday_doc.fillna('-', inplace=True)
monday_doc.replace(to_replace="",  value="-", inplace=True)

monday_doc.reset_index(drop=True, inplace=True)
monday_doc.index.name = 'index'

monday_doc.rename(columns={
                           'repo_link': 'Repository Link',
                           'repo_name': 'Repository Name',
                           'platform_assesment': "Platform's assessment",
                           }, inplace=True)

from datetime import date
## Should I add new/existing and mark for deletion?
today = date.today()
outfile = parent_dir / f"MondayPackageBoard_Update_{today.strftime()}.xlsx"
monday_doc.to_excel(outfile, engine='xlsxwriter', index=True)

178
Index(['id', 'num_variables', 'num_cdes', 'repo_link', 'repo_name',
       'platform_assessment', 'VLMD on Platform', 'CDE mapping on HSS'],
      dtype='object')


In [ ]:
##### DEPRECATED CODE.

## Read the index download, and create a dataset with package links using information from Platform MDS
import json
index_v2 = Path(parent_dir / "DataReuseSummaries/heal-dev-studies-index.json")
with open(index_v2, "r") as f:
    all_lines = f.readlines()
    all_elements_1 = [json.loads(s) for s in all_lines]
dict_1 = {s["_id"]:s for s in all_elements_1}
print(len(dict_1))

import requests

all_studies_dict = []
DEFAULT_MDS_ENDPOINT = 'https://healdata.org/mds/metadata'

cde_col_names = set()

for k in dict_1:
    # k is the HDPID here. 
    new_s = dict_1[k]['_source']
    repos =new_s['metadata']['Data Package Links'] if 'Data Package Links' in new_s['metadata'] else []

    # start creating the data structure
    d = {'id':k, 
         'num_variables':len(new_s['variable_list']), # index
         'num_cdes':len(new_s['section_list']), # index
         'data_available': new_s['metadata']['Data Availability'] if 'Data Availability' in new_s['metadata'] else '', #index, and platform data assessment
         'cdes': new_s['section_list'] #index
         }
    # Create a cde usage indicator data structure in the form of python dict
    cde_dict = {k:1 for k in d['cdes']}
    cde_col_names = cde_col_names.union(set(cde_dict.keys()))


    # Get links from platform that were assessed to be data, and "other_study_websites" from MDS could be something else.
    # We will add these to the data frame as individual links to be matched with platform. 
    # We will add links from the list of other websites only if they are not duplicate from the data package assessment links
    mds_response = requests.get(DEFAULT_MDS_ENDPOINT + '/' + k)
    study_json = mds_response.json()
    gen3_discovery = study_json.get('gen3_discovery', None)
    study_metadata = gen3_discovery.get('study_metadata', None)
    repositories = []
    if study_metadata is not None and ('metadata_location' in study_metadata and 'data_repositories' in study_metadata['metadata_location']):
            repositories = study_metadata['metadata_location']['data_repositories']
    other_website_links = []
    if study_metadata is not None and ('metadata_location' in study_metadata and 'other_study_websites' in study_metadata['metadata_location']):
        other_website_links = study_metadata['metadata_location']['other_study_websites']
    repo_links = [r['repository_study_link'] for r in repositories if 'repository_study_link' in r and len(r['repository_study_link']) > 0]
    other_links = [r for r in other_website_links if r not in repo_links]

    num_repos = len(repo_links) + len(other_links)
    num_data_repos = len(repo_links)
    num_other_repos = len(other_links)

    if len(repo_links) > 0 or len(other_links) > 0:
        # print(repos)
        cnt = 0 
        if len(repo_links) > 0:
            for r in repositories:
                if 'repository_study_link' in r and len(r['repository_study_link']) > 0:
                    all_studies_dict.append({**d, 
                                             'repo_link': r['repository_study_link'], 
                                             'repo_name': r['repository_name'], 
                                             'num_repositories':num_repos, 
                                             'num_data_repositories': num_data_repos, 
                                             'num_other_links': num_other_repos, 
                                             'is_other_website': False, 
                                             **cde_dict})
                    cnt +=1
            if cnt == 0:
                print(f"ERROR: Check HDPID {k} for length of repositories, didnt add any to the dd {len(repositories)}")

        if len(other_links) > 0:
            for r in other_links:
                all_studies_dict.append({**d, 
                                         'repo_link': r, 
                                         'repo_name': '', 
                                         'num_repositories':num_repos, 
                                         'num_data_repositories': num_data_repos, 
                                         'num_other_links': num_other_repos, 
                                         'is_other_website': True, 
                                         **cde_dict})
    else:
            all_studies_dict.append({**d, 
                                     'repo_link': '', 
                                     'repo_name': '', 
                                     'num_repositories': 0, 
                                     'num_other_links': 0,
                                     'num_data_repositories': 0,
                                     **cde_dict })

all_studies_pd = pd.DataFrame(all_studies_dict)
all_studies_pd.to_csv(parent_dir/"DataReuseSummaries/studies_index.csv", index=False)
all_studies_pd['num_repositories'].describe()
print(len(all_studies_dict))
print(f"Number of HDPIDs {len(all_studies_pd['id'].unique())}")

1063


KeyboardInterrupt: 